In [ ]:
# Imports

import os, sys, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from matplotlib.patches import ConnectionPatch
from matplotlib.patches import Rectangle

# Params
RANDOM = False # True: pick a random pair each run; False: deterministic
SEED = 4242 # used when RANDOM == False
POS_ONLY = True # True: only load positive pairs (label==1)
ROW_IDX = 16578 # when RANDOM==False: pick this row index; set to None to use seeded RNG

In [ ]:
# Find repo root (expects folders: models/, utils/, train/)

def find_repo_root(start=None, markers=("models", "utils", "train")):
    p = Path(start or Path.cwd()).resolve()
    for q in (p, *p.parents):
        if all((q / m).exists() for m in markers):
            return q
    raise FileNotFoundError("Start the notebook from inside the project folder (has models/, utils/, train/).")

repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Paths 
pairs_file = repo_root / "data" / "processed" / "pairs_test_dist.jsonl"
raw_img_dir = repo_root / "data" / "raw"
CKPT_PATH = repo_root / "train" / "checkpoints" / "relational" / "model_epoch3.pth"

print("pairs_file:", pairs_file)
print("raw_img_dir:", raw_img_dir)
print("CKPT_PATH:", CKPT_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
eval_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

from models.relational_model import SiameseRelational

In [ ]:
# Load pairs file, apply filters, and select row

assert pairs_file.exists(), f"File not found: {pairs_file}"
assert raw_img_dir.exists(), f"Raw image dir not found: {raw_img_dir}"

with open(pairs_file, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
assert rows, f"{pairs_file} is empty."
df = pd.DataFrame(rows)

if POS_ONLY:
    df = df[df["label"] == 1].reset_index(drop=True)
    assert len(df) > 0, "No pairs left after POS_ONLY filter."

def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if RANDOM:
    ridx = np.random.randint(len(df))
else:
    set_seeds(SEED)
    if ROW_IDX is not None:
        ridx = int(ROW_IDX) % len(df)
    else:
        rng = np.random.default_rng(SEED)
        ridx = int(rng.integers(len(df)))

row = df.iloc[ridx]
print(f"Picked row {ridx} | label={row['label']} | distance={row.get('distance')}")

img1_path = raw_img_dir / row["img1"]
img2_path = raw_img_dir / row["img2"]
assert img1_path.exists(), f"Missing image: {img1_path}"
assert img2_path.exists(), f"Missing image: {img2_path}"
print("img1:", img1_path.name)
print("img2:", img2_path.name)

def load_img(p):
    return Image.open(p).convert("RGB")

img1_pil = load_img(img1_path)
img2_pil = load_img(img2_path)
x1 = eval_tfm(img1_pil).unsqueeze(0).to(device)
x2 = eval_tfm(img2_pil).unsqueeze(0).to(device)


In [ ]:
# Load model and probe feature grid size

model = SiameseRelational(pretrained=False).to(device).eval()
state = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(state, strict=False)
print("Loaded:", CKPT_PATH.name)

with torch.no_grad():
    dummy = torch.zeros(1, 3, 224, 224, device=device)
    fmap = model.encoder(dummy)
    _, _, h, w = fmap.shape
print(f"Feature grid: {h}x{w} (N={h*w})")


In [ ]:
# Attention utilities

def get_attn(model, xq, xk):
    # Runs forward pass with attention output
    #
    # Args:
    #   model: SiameseRelational model
    #   xq (Tensor): query image tensor [1, 3, H, W]
    #   xk (Tensor): key image tensor [1, 3, H, W]
    # Returns:
    #   (Tensor, int, int): attention weights [H, Nq, Nk], feature grid height, feature grid width
    with torch.no_grad():
        out = model(xq, xk, return_attention=True)
    attn = out[3]
    if attn.dim() == 4:
        A = attn.squeeze(0).detach().cpu()
    else:
        A = attn.squeeze(0).unsqueeze(0).cpu()
    f = model.encoder(xq)
    _, _, h, w = f.shape
    return A, h, w

def row_entropy(A, dim=-1, eps=1e-9):
    # Computes entropy along given dimension
    #
    # Args:
    #   A (Tensor): attention weights
    #   dim (int): dimension to sum over
    #   eps (float): small value to avoid log(0)
    # Returns:
    #   Tensor: entropy values
    A = A.clamp_min(eps)
    return -(A * A.log()).sum(dim)

def pick_best_head(A):
    # Selects attention head with lowest mean row entropy
    #
    # Args:
    #   A (Tensor): attention weights [H, Nq, Nk] or [Nq, Nk]
    # Returns:
    #   (int, ndarray): best head index, entropies per head
    if A.dim() == 2:
        A = A.unsqueeze(0)
    ent = row_entropy(A, dim=-1).mean(dim=-1)
    idx = int(ent.argmin().item())
    return idx, ent.cpu().numpy()

def upsample_map(m_hw, out_wh):
    # Rescales attention map to match image resolution
    #
    # Args:
    #   m_hw (ndarray): attention map [h, w]
    #   out_wh (tuple): output size (H, W)
    # Returns:
    #   ndarray: upsampled map
    h, w = m_hw.shape
    H, W = out_wh
    t = torch.tensor(m_hw)[None, None]
    t = (t - t.min()) / (t.max() - t.min() + 1e-9)
    t = F.interpolate(t, size=(H, W), mode="bilinear", align_corners=False)
    return t.squeeze().cpu().numpy()

def plot_attention_maps(A, key_img_pil, h, w, n_side=4, title_prefix="1→2"):
    # Plots attention heatmaps for multiple query positions
    #
    # Args:
    #   A (Tensor): attention weights [Nq, Nk]
    #   key_img_pil (PIL.Image): key image
    #   h, w (int): feature grid height and width
    #   n_side (int): queries per axis
    #   title_prefix (str): label for plots
    # Returns:
    #   None
    Nq, Nk = A.shape
    Hpix, Wpix = key_img_pil.height, key_img_pil.width
    qs = []
    for rr in torch.linspace(0, h - 1, n_side).round().long():
        for cc in torch.linspace(0, w - 1, n_side).round().long():
            qs.append(int(rr * w + cc))
    cols = len(qs) + 1
    fig, axes = plt.subplots(1, cols, figsize=(1.6 * cols, 1.6))
    axes[0].imshow(key_img_pil)
    axes[0].axis("off")
    axes[0].set_title("keys")
    for i, q in enumerate(qs, start=1):
        m = A[q].reshape(h, w)
        hm = upsample_map(m, (Hpix, Wpix))
        axes[i].imshow(key_img_pil)
        axes[i].imshow(hm, cmap="jet", alpha=0.55)
        axes[i].set_title(f"{title_prefix} q={q}")
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()

def plot_correspondence_arrows(A, img_q_pil, img_k_pil, h, w, k=32, use_centroid=True, title="1→2"):
    # Draws correspondence lines between query and key positions
    #
    # Args:
    #   A (Tensor): attention weights [Nq, Nk]
    #   img_q_pil, img_k_pil (PIL.Image): query and key images
    #   h, w (int): feature grid height and width
    #   k (int): number of correspondences
    #   use_centroid (bool): choose key via centroid instead of argmax
    #   title (str): plot title
    # Returns:
    #   None
    Nq, Nk = A.shape
    H1, W1 = img_q_pil.height, img_q_pil.width
    H2, W2 = img_k_pil.height, img_k_pil.width

    def idx_to_xy(idx, H, W, Hpix, Wpix):
        r, c = divmod(int(idx), W)
        return (c + 0.5) * (Wpix / W), (r + 0.5) * (Hpix / H)

    qs = list(range(0, h * w, max(1, (h * w) // k)))[:k]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
    ax1.imshow(img_q_pil)
    ax1.axis("off")
    ax1.set_title("queries")
    ax2.imshow(img_k_pil)
    ax2.axis("off")
    ax2.set_title("keys")

    for q in qs:
        wq = A[q]
        if use_centroid:
            m = (wq / (wq.sum() + 1e-9)).reshape(h, w).cpu().numpy()
            ys, xs = np.mgrid[0:h, 0:w]
            cx, cy = (m * xs).sum(), (m * ys).sum()
            k_idx = int(round(cy)) * w + int(round(cx))
        else:
            k_idx = int(wq.argmax().item())
        xq, yq = idx_to_xy(q, h, w, H1, W1)
        xk, yk = idx_to_xy(k_idx, h, w, H2, W2)
        ax1.plot(xq, yq, "o", ms=3)
        ax2.plot(xk, yk, "o", ms=3)
        con = ConnectionPatch((xk, yk), (xq, yq), "data", "data", axesA=ax2, axesB=ax1, lw=0.7, alpha=0.85)
        ax2.add_artist(con)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# Grid indexing and visualization utilities

def idx_to_rc(idx, w):
    # Converts flat index to (row, col)
    #
    # Args:
    #   idx (int): flat index
    #   w (int): grid width
    # Returns:
    #   (int, int): row, col
    return int(idx) // w, int(idx) % w

def rc_to_idx(r, c, w):
    # Converts (row, col) to flat index
    #
    # Args:
    #   r (int): row index
    #   c (int): column index
    #   w (int): grid width
    # Returns:
    #   int: flat index
    return r * w + c

def draw_index_grid(img_pil, h, w, title="", fontsize=9, edge_alpha=0.6, text_bg=True):
    # Draws grid overlay with flat indices
    #
    # Args:
    #   img_pil (PIL.Image): image to overlay
    #   h, w (int): grid height and width
    #   title (str): plot title
    #   fontsize (int): index label size
    #   edge_alpha (float): grid line transparency
    #   text_bg (bool): draw background for text
    # Returns:
    #   None
    Hpix, Wpix = img_pil.height, img_pil.width
    rh, rw = Hpix / h, Wpix / w

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(img_pil)
    ax.set_title(title)
    ax.axis("off")

    for r in range(h):
        for c in range(w):
            x0, y0 = c * rw, r * rh
            rect = Rectangle((x0, y0), rw, rh, fill=False, linewidth=1.2, alpha=edge_alpha)
            ax.add_patch(rect)
            idx = r * w + c
            xc = x0 + rw / 2
            yc = y0 + rh / 2
            bbox = dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.6) if text_bg else None
            ax.text(xc, yc, str(idx), ha="center", va="center", fontsize=fontsize, bbox=bbox)

    plt.tight_layout()
    plt.show()

def draw_index_grid_coords(img_pil, h, w, title="", fontsize=9, edge_alpha=0.6):
    # Draws grid overlay with (row, col) labels
    #
    # Args:
    #   img_pil (PIL.Image): image to overlay
    #   h, w (int): grid height and width
    #   title (str): plot title
    #   fontsize (int): label font size
    #   edge_alpha (float): grid line transparency
    # Returns:
    #   None
    Hpix, Wpix = img_pil.height, img_pil.width
    rh, rw = Hpix / h, Wpix / w

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(img_pil)
    ax.set_title(title)
    ax.axis("off")

    for r in range(h):
        for c in range(w):
            x0, y0 = c * rw, r * rh
            rect = Rectangle((x0, y0), rw, rh, fill=False, linewidth=1.2, alpha=edge_alpha)
            ax.add_patch(rect)
            xc = x0 + rw / 2
            yc = y0 + rh / 2
            ax.text(xc, yc, f"({r},{c})", ha="center", va="center", fontsize=fontsize,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.6))

    plt.tight_layout()
    plt.show()


In [ ]:
# Visualize top-k key positions for a given query

def plot_query_topk_on_keys(A_row, img_key_pil, h, w, k=10, title="Top-k keys for query"):
    # Highlights top-k key positions for a single query
    #
    # Args:
    #   A_row (Tensor or ndarray): attention weights for one query [Nk]
    #   img_key_pil (PIL.Image): key image
    #   h, w (int): feature grid height and width
    #   k (int): number of keys to show
    #   title (str): plot title
    # Returns:
    #   None
    if isinstance(A_row, torch.Tensor):
        a = A_row.detach().cpu().float().numpy()
    else:
        a = np.asarray(A_row, dtype=np.float32)

    Hpix, Wpix = img_key_pil.height, img_key_pil.width
    rh, rw = Hpix / h, Wpix / w

    topk_idx = np.argsort(-a)[:k]
    topk_wts = a[topk_idx]
    topk_wts = topk_wts / (topk_wts.sum() + 1e-9)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(img_key_pil)
    ax.axis("off")
    ax.set_title(title)

    for rank, idx in enumerate(topk_idx):
        r, c = idx_to_rc(idx, w)
        x0, y0 = c * rw, r * rh
        rect = Rectangle((x0, y0), rw, rh, fill=False, linewidth=2.0, alpha=0.9)
        ax.add_patch(rect)
        xc = x0 + rw / 2
        yc = y0 + rh / 2
        ax.text(xc, yc, f"{idx}\n{topk_wts[rank]:.2f}", ha="center", va="center",
                fontsize=9, bbox=dict(boxstyle="round,pad=0.2", fc="yellow", ec="black", alpha=0.7))

    plt.tight_layout()
    plt.show()


In [ ]:
# Load selected images, transform, and compute attention in both directions

img1_path = raw_img_dir / row["img1"]
img2_path = raw_img_dir / row["img2"]
img1_pil = load_img(img1_path)
img2_pil = load_img(img2_path)

x1 = eval_tfm(img1_pil).unsqueeze(0).to(device)
x2 = eval_tfm(img2_pil).unsqueeze(0).to(device)

with torch.no_grad():
    A12_all, h, w = get_attn(model, x1, x2)
    A21_all, _, _ = get_attn(model, x2, x1)

h12, ent12 = pick_best_head(A12_all)
h21, ent21 = pick_best_head(A21_all)
A12 = A12_all[h12]
A21 = A21_all[h21]


In [ ]:
# Visualize feature grid indexing for both images

draw_index_grid(img1_pil, h, w, title=f"img1 grid, {h}×{w}")
draw_index_grid(img2_pil, h, w, title=f"img2 grid, {h}×{w}")


In [ ]:
# Plot Attention Mapts

plot_attention_maps(A12, img2_pil, h, w, n_side=4, title_prefix="1→2")
plot_attention_maps(A21, img1_pil, h, w, n_side=4, title_prefix="2→1")

In [ ]:
# Plot Correspondence Arrows

plot_correspondence_arrows(A12, img1_pil, img2_pil, h, w, k=40, use_centroid=True, title="1→2 correspondences")
plot_correspondence_arrows(A21, img2_pil, img1_pil, h, w, k=40, use_centroid=True, title="2→1 correspondences")